# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

_Tuliskan pemahamanmu terhadap use case di sini (objective, target/label, batasan/asumsi)._

Membangun fitur otomatis untuk **mengklasifikasikan sentimen ulasan pelanggan** (positif/negatif) pada halaman produk e-commerce. Fitur ini ditujukan untuk membantu tim produk memahami kepuasan pelanggan secara real-time dan mengidentifikasi produk yang bermasalah lebih cepat.

Sebelum tim memutuskan pendekatan mana yang akan dipakai di sistem produksi, eksperimen ini membandingkan dua kandidat pendekatan:

1. **Model klasik Machine Learning** — dilatih sendiri menggunakan Scikit-learn (TF-IDF + Logistic Regression).
2. **LLM API (Gemini)** — pendekatan prompting tanpa training.

Perbandingan dilakukan pada dataset dan test set yang sama, dievaluasi dengan metrik Accuracy, Precision, Recall, dan F1-Score.

### 1.2 Definisi Target / Label

- **Target:** Klasifikasi biner sentimen ulasan.
- **Label:** `positif` atau `negatif`.
- **Input (fitur):** Kolom `review_text` — teks ulasan pelanggan.
- **Output:** Prediksi label sentimen untuk setiap ulasan.

### 1.3 Dataset

- **File:** `customer_reviews_sentiment.csv`
- **Jumlah baris:** 200
- **Kolom:** `review_id`, `product_name`, `review_text`, `sentiment`
- **Distribusi label:** ~55% positif, ~45% negatif (cukup seimbang)
- **Bahasa:** Indonesia (campuran formal & informal)

### 1.4 Batasan & Asumsi

**Batasan:**
- Dataset terbatas (200 baris) dan mengandung banyak `review_text` yang duplikat antar baris produk — hanya ~40–50 kalimat unik.
- Ulasan hanya dalam bahasa Indonesia, tidak mencakup bahasa daerah atau campuran kode.
- Eksperimen dijalankan pada free tier Gemini API, sehingga ada rate limit dan kemungkinan 503/429.
- Test set yang dipakai untuk kedua pendekatan harus identik agar perbandingan adil.

**Asumsi:**
- Setiap ulasan diasumsikan memiliki **satu sentimen dominan** (tidak ada label multi-sentimen).
- Label ground truth pada dataset dianggap benar.
- Untuk menghindari *data leakage* akibat duplikasi review, digunakan **GroupShuffleSplit** — semua baris dengan `review_text` yang sama masuk ke sisi train atau test yang sama, tidak dicampur.
- Perbandingan fokus pada **kualitas prediksi**, bukan pada optimasi infrastruktur produksi (misal caching, batching).

## 2. Import Library & Load Dataset

In [11]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types, errors
import os
import time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

df = pd.read_csv('../data/customer_reviews_sentiment.csv')
df.head()

,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

Digunakan GroupShuffleSplit dikarenakan banyaknya duplikat pada review text namun pada product name yang berbeda, jika tidak digunakan split tersebut maka akan ada leakage pada model klasik

In [12]:
from sklearn.model_selection import GroupShuffleSplit

X = df['review_text']
y = df['sentiment']
groups = df['review_text']   # grup = kalimat review

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# Verifikasi tidak ada overlap
overlap = set(X_train) & set(X_test)
print("Overlap review_text:", len(overlap))   # harus 0

Train size: 159
Test size: 41
Overlap review_text: 0


## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [13]:
vectorizer = TfidfVectorizer(lowercase=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Training Model

In [14]:
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_vec, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

### 4.3 Prediksi pada Test Set

In [15]:
y_pred_classic = clf.predict(X_test_vec)

## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API

In [16]:
# Simpan API key di environment variable, JANGAN hardcode langsung di notebook
load_dotenv()
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

### 5.2 Merancang Prompt

_Tuliskan prompt yang kamu rancang di sini, dan jelaskan alasannya (zero-shot / few-shot)._

Zero-shot digunakan untuk menggeneralisasi tujuan dari berbagai bentuk input

In [20]:
MODELS = [
    "gemini-3.1-flash-lite"
]


def classify_sentiment_llm(review_text, max_retries=3):
    prompt = f'''Klasifikasikan sentimen ulasan berikut.
Balas HANYA dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:'''

    for model_name in MODELS:
        for attempt in range(max_retries):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(temperature=0.2)
                )
                return response.text.strip().lower()
            except errors.ServerError:
                time.sleep(2 ** attempt)
            except errors.ClientError as e:
                msg = str(e)
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                    time.sleep(5 * (attempt + 1))
                    continue
                break
    return None

### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).


> Karena Gemini API sedang overload dan deadline dekat, digunakan:
- **ThreadPoolExecutor** dengan 5 worker untuk paralelisasi.
- **Fallback antar model** kalau satu model 503.
- **Retry** untuk 503 dan 429.
- **Timeout global** 170 detik agar tidak stuck.

In [21]:
def normalize_prediction(raw_output):
    if raw_output is None:
        return "negatif"
    text = raw_output.strip().lower()
    if any(kw in text for kw in ["positif", "positive"]):
        return "positif"
    if any(kw in text for kw in ["negatif", "negative"]):
        return "negatif"
    return "negatif"

In [19]:
# import time
#
# sample = "Produk ini bagus banget, saya suka!"
# prompt = f'''Klasifikasikan sentimen ulasan berikut.
# Balas HANYA dengan satu kata: positif atau negatif.
#
# Ulasan: "{sample}"
# Sentimen:'''
#
# models_to_test = [
#     # "gemini-3.8-flash",
#     # "gemini-3.7-flash",
#     # "gemini-3.6-flash",
#     # "gemini-3.5-flash",
#     "gemini-3.5-flash-lite",
#     "gemini-3.1-flash-lite",
#     "gemini-flash-lite-latest",
#     "gemini-flash-latest",
# ]
#
# for model_name in models_to_test:
#     print(f"\n=== {model_name} ===")
#     t0 = time.time()
#     try:
#         response = client.models.generate_content(
#             model=model_name,
#             contents=prompt,
#             config=types.GenerateContentConfig(temperature=0.2)
#         )
#         elapsed = time.time() - t0
#         print(f"SUCCESS ({elapsed:.1f}s): {repr(response.text)}")
#     except Exception as e:
#         elapsed = time.time() - t0
#         print(f"ERROR ({elapsed:.1f}s): {type(e).__name__}")
#         print(f"  MSG: {str(e)[:200]}")


=== gemini-3.5-flash-lite ===
SUCCESS (101.8s): 'positif'

=== gemini-3.1-flash-lite ===
SUCCESS (1.1s): 'Positif'

=== gemini-flash-lite-latest ===


KeyboardInterrupt: 

In [22]:
def process_one(args):
    idx, review = args
    raw = classify_sentiment_llm(review)
    return idx, normalize_prediction(raw)


y_pred_llm = [None] * len(X_test)
tasks = list(enumerate(X_test))

START_TIME = time.time()
MAX_TOTAL_SECONDS = 300

executor = ThreadPoolExecutor(max_workers=2)
futures = {executor.submit(process_one, t): t[0] for t in tasks}

try:
    for future in as_completed(futures, timeout=MAX_TOTAL_SECONDS):
        try:
            idx, pred = future.result()
        except Exception as e:
            print(f"Future error: {type(e).__name__}")
            continue
        y_pred_llm[idx] = pred
        filled = sum(p is not None for p in y_pred_llm)
        print(f"{filled}/{len(X_test)} dalam {time.time() - START_TIME:.1f}s")
except TimeoutError:
    print("Timeout — membatalkan sisa future")

executor.shutdown(wait=False, cancel_futures=True)

print(f"Total waktu: {time.time() - START_TIME:.1f}s")
print("Distribusi sementara:", Counter(y_pred_llm))
print("Jumlah None:", sum(p is None for p in y_pred_llm))


none_indices = [i for i, p in enumerate(y_pred_llm) if p is None]
print(f"Retry {len(none_indices)} request yang gagal, sequential")

RETRY_START = time.time()
for j, i in enumerate(none_indices):
    if time.time() - RETRY_START > 180:
        print("Timeout retry")
        break
    review = X_test.iloc[i] if hasattr(X_test, "iloc") else X_test[i]
    raw = classify_sentiment_llm(review)
    y_pred_llm[i] = normalize_prediction(raw)
    print(f"{j+1}/{len(none_indices)} [{i}] -> {y_pred_llm[i]}")

y_pred_llm = [p if p is not None else "negatif" for p in y_pred_llm]
print("\nFinal distribusi:", Counter(y_pred_llm))
print("None tersisa:", sum(p is None for p in y_pred_llm))

1/41 dalam 1.1s
2/41 dalam 2.0s
3/41 dalam 3.1s
4/41 dalam 4.0s
5/41 dalam 5.0s
6/41 dalam 5.9s
7/41 dalam 6.7s
8/41 dalam 7.8s
9/41 dalam 9.0s
10/41 dalam 10.2s
11/41 dalam 11.0s
12/41 dalam 11.9s
13/41 dalam 12.9s
14/41 dalam 14.0s
15/41 dalam 15.4s
16/41 dalam 16.4s
17/41 dalam 46.3s
18/41 dalam 47.3s
19/41 dalam 63.4s
20/41 dalam 63.7s
21/41 dalam 64.3s
22/41 dalam 65.1s
23/41 dalam 65.6s
24/41 dalam 66.2s
25/41 dalam 66.9s
26/41 dalam 67.1s
27/41 dalam 68.0s
28/41 dalam 68.0s
29/41 dalam 68.9s
30/41 dalam 69.2s
31/41 dalam 69.8s
32/41 dalam 70.2s
33/41 dalam 70.7s
34/41 dalam 101.2s
35/41 dalam 101.6s
36/41 dalam 118.1s
37/41 dalam 118.5s
38/41 dalam 119.5s
39/41 dalam 119.5s
40/41 dalam 120.5s
41/41 dalam 120.8s
Total waktu: 120.8s
Distribusi sementara: Counter({'positif': 24, 'negatif': 17})
Jumlah None: 0
Retry 0 request yang gagal, sequential

Final distribusi: Counter({'positif': 24, 'negatif': 17})
None tersisa: 0


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [23]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

=== Model Klasik (Scikit-learn) ===
Accuracy: 0.5121951219512195
              precision    recall  f1-score   support

     negatif       0.38      0.64      0.47        14
     positif       0.71      0.44      0.55        27

    accuracy                           0.51        41
   macro avg       0.54      0.54      0.51        41
weighted avg       0.59      0.51      0.52        41

Confusion Matrix:
 [[ 9  5]
 [15 12]]


### 6.2 Evaluasi LLM API

In [24]:
print('=== LLM API (Gemini) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_llm))
print(classification_report(y_test, y_pred_llm))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_llm))

=== LLM API (Gemini) ===
Accuracy: 0.926829268292683
              precision    recall  f1-score   support

     negatif       0.82      1.00      0.90        14
     positif       1.00      0.89      0.94        27

    accuracy                           0.93        41
   macro avg       0.91      0.94      0.92        41
weighted avg       0.94      0.93      0.93        41

Confusion Matrix:
 [[14  0]
 [ 3 24]]


### 6.3 Tabel Perbandingan Ringkasan

_Susun tabel ringkasan (bisa markdown table atau DataFrame) yang membandingkan Accuracy, Precision, Recall, F1-Score kedua pendekatan._

In [25]:
from sklearn.metrics import precision_score, recall_score, f1_score
def get_metrics(y_true, y_pred, name):
    return {
        "Pendekatan": name,
        "Accuracy":  round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "Recall":    round(recall_score(y_true, y_pred, average="weighted", zero_division=0), 4),
        "F1-Score":  round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
    }

comparison = pd.DataFrame([
    get_metrics(y_test, y_pred_classic, "Model Klasik (TF-IDF + LogReg)"),
    get_metrics(y_test, y_pred_llm,    "LLM API (Gemini, zero-shot)"),
])
comparison

,Pendekatan,Accuracy,Precision,Recall,F1-Score
0,Model Klasik (TF-IDF + LogReg),0.5122,0.5929,0.5122,0.5209
1,"LLM API (Gemini, zero-shot)",0.9268,0.9397,0.9268,0.9282


## 7. Analisis Trade-off dan Limitation

_Tuliskan analisis: pendekatan mana yang lebih unggul dari sisi performa, trade-off effort/kecepatan/biaya, dan keterbatasan masing-masing pendekatan._

Berdasarkan hasil evaluasi pada test set yang sama (41 review, GroupShuffleSplit untuk menghindari kebocoran data):

| Metrik | Model Klasik | LLM API (Gemini) |
|--------|--------------|------------------|
| Accuracy | 0.512 | **0.927** |
| Precision (weighted) | 0.59 | **0.94** |
| Recall (weighted) | 0.51 | **0.93** |
| F1-Score (weighted) | 0.52 | **0.93** |

**LLM API (Gemini) jauh lebih unggul** di semua metrik, dengan selisih accuracy ~41 poin persentase. Confusion matrix menunjukkan model klasik banyak menghasilkan *false negative* (15 review positif diprediksi negatif), sedangkan LLM hanya salah pada 3 dari 27 review positif.

Temuan penting: pada percobaan awal dengan split acak, model klasik mencapai accuracy 1.00. Setelah menggunakan **GroupShuffleSplit** (memastikan review yang sama tidak muncul di train dan test), performa model klasik turun drastis ke 0.51. Ini menunjukkan skor sempurna sebelumnya adalah artefak *data leakage*, bukan cerminan kemampuan generalisasi model.

| Aspek | Model Klasik (TF-IDF + LogReg) | LLM API (Gemini, zero-shot) |
|-------|-------------------------------|------------------------------|
| **Effort implementasi** | Perlu training, tuning hyperparameter, feature engineering | Tanpa training, cukup rancang prompt |
| **Kecepatan training** | Sangat cepat (< 5 detik untuk 160 baris) | Tidak ada training |
| **Kecepatan inference** | Sangat cepat (< 1 detik untuk 41 review) | ~2 menit untuk 41 review (dengan paralelisasi 2 worker) |
| **Biaya** | Gratis (compute lokal) | Free tier terbatas; produksi berbayar per token |
| **Skalabilitas** | Sangat baik untuk volume besar | Bergantung kuota & rate limit API |
| **Maintenance** | Perlu retrain jika distribusi data berubah | Tidak perlu retrain, cukup update prompt |
| **Konsistensi output** | Deterministik | Perlu normalisasi (output bisa EN/ID) |


## 8. Rekomendasi Technical Approach

_Tuliskan rekomendasi akhir beserta alasannya, berdasarkan hasil eksperimen di atas._

Berdasarkan hasil eksperimen, **LLM API (Gemini) direkomendasikan sebagai pendekatan utama** untuk use case klasifikasi sentimen ulasan pelanggan ini.

### Alasan Utama

1. **Performa jauh lebih unggul secara terukur.** LLM mencapai Accuracy 0.93 dan F1-Score 0.93, sementara model klasik hanya 0.51 dan 0.52 pada test set yang sama. Selisih ini signifikan secara praktis — LLM salah pada 3 dari 41 review, model klasik salah pada 20 dari 41.

2. **Tidak memerlukan training atau feature engineering.** LLM dapat langsung digunakan dengan prompt zero-shot, menghemat waktu pengembangan dan tidak memerlukan dataset berlabel besar.

3. **Lebih tahan terhadap variasi bahasa.** LLM sudah memiliki pengetahuan bahasa dari pretraining, sehingga lebih baik dalam menangani variasi kalimat, sinonim, dan gaya bahasa yang tidak muncul di training set.

4. **Iterasi cepat.** Untuk menyesuaikan dengan domain baru, cukup ubah prompt — tidak perlu retrain model.
### Kapan Model Klasik Tetap Relevan

Model klasik (TF-IDF + LogReg) tetap layak dipertimbangkan sebagai **komplementer** ketika:
- Volume prediksi sangat tinggi dan biaya API menjadi kendala utama.
- Latency rendah (< 1 detik) menjadi syarat mutlak.
- Data training sudah besar, beragam, dan representatif — pada kondisi ini model klasik bisa mencapai performa mendekati LLM dengan biaya jauh lebih murah.